# Stepageddon - Train Step Chart Model

Train a neural network to generate DDR step charts from audio.

**Prerequisites:**
1. Run `prepare_data.py` locally to preprocess your charts
2. Zip the training data: `cd backend/ml && zip -r training_data.zip training_data/`
3. Upload `training_data.zip` directly to Colab when prompted (cell below)

**Runtime:** Select GPU (T4) under Runtime > Change runtime type

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# Install dependencies
!pip install -q librosa numpy torch simfile pydantic-settings asyncpg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 91.2 MB/s eta 0:00:00


In [8]:
# Pull the source from Google Drive into /content so imports resolve.
#
# Expected Drive layout (adjust SRC if yours differs):
#   /content/drive/MyDrive/stepageddon/backend/
#       ml/         (model.py, dataset.py, train.py, prepare_data.py, ...)
#       services/   (sm_parser.py used by prepare_data; safe to skip if you
#                    only need ml.train at runtime, but harmless to copy)
#       modules/    (step_generator schemas — required by inference.py)
#
# We copy these to /content/{ml,services,modules}. Cell 7 adds /content to
# sys.path so `import ml.train`, `import services.sm_parser`, and
# `import modules.step_generator...` all work without any package install.
import os, shutil
from pathlib import Path

SRC = Path('/content/drive/MyDrive/stepageddon')
DST = Path('/content')

assert SRC.exists(), f'Source not found: {SRC}. Adjust SRC to your Drive layout.'

for pkg in ('ml', 'src'):
    src_pkg = SRC / pkg
    dst_pkg = DST / pkg
    if not src_pkg.exists():
        print(f'  skip {pkg}/ (not in Drive)')
        continue
    if dst_pkg.exists():
        shutil.rmtree(dst_pkg)
    # dirs_exist_ok unused since we cleaned above; copytree handles symlinks fine.
    shutil.copytree(src_pkg, dst_pkg)
    # Make sure each top-level dir is a real Python package.
    init_file = dst_pkg / '__init__.py'
    if not init_file.exists():
        init_file.touch()
    print(f'  copied {pkg}/  ({sum(1 for _ in dst_pkg.rglob("*.py"))} .py files)')

# Sanity-check the ml/ contents we actually need to train.
for needed in ('ml/model.py', 'ml/dataset.py', 'ml/train.py', 'ml/prepare_data.py'):
    assert (DST / needed).exists(), f'Missing {needed} after copy'
print('\nAll required modules in place.')
!ls /content/ml/


  copied ml/  (11 .py files)
  copied src/  (16 .py files)

All required modules in place.
choreography_metrics.py  inference.py  patterns.py	style_profiles.py
dataset.py		 __init__.py   prepare_data.py	train_colab.ipynb
eval_choreography.py	 model.py      sm_parser.py	train.py


In [4]:
# Upload and extract training data
# Pick ONE option below and comment out the others

import zipfile, os
ZIP_PATH = '/content/drive/MyDrive/stepageddon/training_data.zip'  # adjust path if needed
print(f'Copying and extracting from Drive...')
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall('/content/')
print('Done!')

!ls /content/training_data/ | head -20

Copying and extracting from Drive...
Done!
manifest.json
song_0000.npz
song_0001.npz
song_0002.npz
song_0004.npz
song_0005.npz
song_0006.npz
song_0007.npz
song_0008.npz
song_0009.npz
song_0010.npz
song_0011.npz
song_0012.npz
song_0013.npz
song_0014.npz
song_0015.npz
song_0016.npz
song_0017.npz
song_0018.npz
song_0019.npz


In [ ]:
# Verify data is accessible and carries v8 per-arrow labels (arrow_labels_<diff>).
import json
from pathlib import Path

DATA_DIR = Path('/content/training_data')
CHECKPOINT_DIR = Path('/content/drive/MyDrive/stepageddon/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

manifest_path = DATA_DIR / 'manifest.json'
with open(manifest_path) as f:
    manifest = json.load(f)

entries = manifest['entries']
print(f'Total training examples: {len(entries)}')
print(f'Per-channel feat stats: n_in_channels={manifest["n_in_channels"]} '
      f'mel_mean range=[{min(manifest["feat_mean"][:80]):.4f}, {max(manifest["feat_mean"][:80]):.4f}]')

from collections import Counter
diff_counts = Counter(e['difficulty'] for e in entries)
print(f'Difficulty distribution: {dict(diff_counts)}')

# Check a sample file — v8 requires arrow_labels_<diff> [T,4] to train the head.
import numpy as np
sample_entry = entries[0]
sample = np.load(DATA_DIR / sample_entry['filename'])
labels = sample[sample_entry['labels_key']]
arrow_key = sample_entry['arrow_labels_key']
assert arrow_key in sample.files, (
    f"{arrow_key} missing — re-run prepare_data.py (FORMAT_VERSION>=8) to train v8."
)
arrows = sample[arrow_key]
print(f"\nSample: {sample_entry['song_title']} ({sample_entry['difficulty']})")
print(f'  feats shape: {sample["feats"].shape}')
print(f'  Labels shape: {labels.shape}')
print(f'  arrow_labels shape: {arrows.shape}  (per-column L/D/U/R onset indicators)')


In [6]:
# Add to Python path
import sys
sys.path.insert(0, '/content')


In [ ]:
# Build dataloaders, model, optimizer, loss using train.py helpers.
# v8 hybrid: 3 dense heads (onset / sustain / intensity) PLUS a learned per-arrow
# direction head (L/D/U/R), on an 88-channel audio feature tensor (mel +
# onset_strength + spectral_contrast). The direction head is supervised only on
# onset frames (multi-label BCE) and is consumed at inference as a *preference*
# that the foot-flow playability constraints then filter (model proposes,
# constraints dispose). arrow_weight scales the direction loss term.
#
# The training data (arrow_labels_<diff> [T,4]) already carries the per-column
# labels, so no re-run of prepare_data is needed for v8.
#
# NOTE: make sure the ml/*.py copied from Drive are the v8 versions (model.py,
# dataset.py, train.py, inference.py) — re-upload them to Drive before running.

from types import SimpleNamespace
import torch
from torch.amp import GradScaler
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

from ml.train import (
    build_dataloaders, build_model, build_optimizer_and_scheduler,
    build_loss, seed_everything,
)

# Hyperparameters — keep in sync with ml.train.build_argparser defaults.
args = SimpleNamespace(
    data_dir=str(DATA_DIR),
    checkpoint_dir=str(CHECKPOINT_DIR),
    epochs=50,
    batch_size=32,
    lr=3e-4,
    hidden_dim=160,
    n_heads=4,
    n_layers=3,
    tcn_dilations='1,2,4,8,16',
    chunk_frames=500,        # 5s at ~100fps
    val_split=0.1,
    weight_decay=0.01,
    warmup_epochs=2.0,
    focal_gamma=2.0,
    sustain_weight=0.5,
    intensity_weight=0.5,
    arrow_weight=1.0,         # v8 learned direction (per-column) BCE weight
    onset_pos_weight=None,    # default sqrt((1-p)/p) capped at 50
    sustain_pos_weight=None,  # default sqrt((1-p)/p) capped at 20
    p_density_swap=0.5,
    intro_outro_oversample_prob=0.15,
    plateau_patience=4,
    plateau_factor=0.5,
    dropout=0.1,
    num_workers=2,
    ema_decay=0.999,
    seed=42,
    tol_frames=3,            # ~30ms at 100fps
    log_interval=50,
    difficulty=None,
    device='cuda',
    max_songs=None,          # cap entries for a smoke run; None = full dataset
)

seed_everything(args.seed)
device = torch.device('cuda')

built = build_dataloaders(args)
n_in_channels = built.full_dataset.n_in_channels
model = build_model(
    args, built.onset_prior, built.sustain_prior,
    n_in_channels=n_in_channels, device=device,
)
criterion = build_loss(built.onset_prior, built.sustain_prior, args).to(device)
optimizer, scheduler = build_optimizer_and_scheduler(
    model, args, steps_per_epoch=max(1, len(built.train_loader)),
)
scaler = GradScaler('cuda')
ema_model = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(args.ema_decay))

print(f'Train entries: {len(built.train_idx)}  Val entries: {len(built.val_idx)}')
print(f'Empirical onset prior: {built.onset_prior:.6f}')
print(f'Empirical sustain prior: {built.sustain_prior:.6f}')
print(f'Empirical density/difficulty: {built.default_density_by_id.tolist()}')
print(f'Arrow head present: {model.arrow_head is not None}  arrow_weight={args.arrow_weight}')


In [ ]:
# Training loop — calls into ml.train.train_one_epoch / validate.
# Auto-resumes from last_model.pt on Drive if it exists.
import time
from ml.train import train_one_epoch, validate, save_checkpoint, composite_score

history = {'train_loss': [], 'val_loss': [], 'onset_tol_f1': [], 'sustain_f1': [],
           'arrow_col_f1': [], 'composite': []}
best_composite = -1.0
start_epoch = 0

EPOCH_LOG_PATH = CHECKPOINT_DIR / 'epochs.txt'

last_ckpt_path = CHECKPOINT_DIR / 'last_model.pt'
if last_ckpt_path.exists():
    print(f'Resuming from {last_ckpt_path}')
    ckpt = torch.load(last_ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    if ckpt.get('ema_state_dict') is not None:
        ema_model.module.load_state_dict(ckpt['ema_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_composite = composite_score(ckpt.get('metrics', {}))
    print(f'  -> resumed at epoch {start_epoch}, best composite={best_composite:.3f}')
else:
    print('No checkpoint found — starting fresh from epoch 0.')

for epoch in range(start_epoch, args.epochs):
    t0 = time.time()
    train_metrics = train_one_epoch(
        model, built.train_loader, criterion, optimizer, scheduler,
        scaler, device, ema_model=ema_model,
        log_interval=args.log_interval,
    )
    val_metrics = validate(
        ema_model.module, built.val_loader, criterion, device,
        tol_frames=args.tol_frames,
    )

    elapsed = time.time() - t0
    lr = optimizer.param_groups[0]['lr']
    composite = composite_score(val_metrics)
    history['train_loss'].append(train_metrics['loss'])
    history['val_loss'].append(val_metrics['loss'])
    history['onset_tol_f1'].append(val_metrics['onset_tol_f1'])
    history['sustain_f1'].append(val_metrics['sustain_f1'])
    history['arrow_col_f1'].append(val_metrics.get('arrow_col_f1', 0.0))
    history['composite'].append(composite)

    epoch_line = (
        f"Epoch {epoch+1:3d}/{args.epochs} | "
        f"train={train_metrics['loss']:.4f} "
        f"(onset={train_metrics['onset_loss']:.4f} "
        f"sustain={train_metrics['sustain_loss']:.4f} "
        f"intensity={train_metrics['intensity_loss']:.4f} "
        f"arrow={train_metrics.get('arrow_loss', 0.0):.4f}) | "
        f"val={val_metrics['loss']:.4f} "
        f"onset_tol_f1={val_metrics['onset_tol_f1']:.3f} "
        f"f1_50ms={val_metrics['onset_tol_f1_50ms']:.3f} "
        f"sustain_f1={val_metrics['sustain_f1']:.3f} "
        f"sustain_iou={val_metrics['sustain_iou']:.3f} "
        f"intensity_auc={val_metrics['intensity_jump_auc']:.3f} "
        f"arrow_col_f1={val_metrics.get('arrow_col_f1', 0.0):.3f} "
        f"intensity_mse={val_metrics['intensity_mse']:.4f} "
        f"| composite={composite:.3f} "
        f"| lr={lr:.2e} | {elapsed:.1f}s"
    )
    print(epoch_line, flush=True)
    with open(EPOCH_LOG_PATH, 'a') as f:
        f.write(epoch_line + '\n')

    save_checkpoint(
        CHECKPOINT_DIR / 'last_model.pt', epoch, model, ema_model,
        optimizer, scheduler, scaler, val_metrics,
        built.default_density_by_id,
        built.full_dataset.feat_mean, built.full_dataset.feat_std,
        n_in_channels, built.onset_prior, built.sustain_prior,
        args,
    )
    if composite > best_composite:
        best_composite = composite
        save_checkpoint(
            CHECKPOINT_DIR / 'best_model.pt', epoch, model, ema_model,
            optimizer, scheduler, scaler, val_metrics,
            built.default_density_by_id,
            built.full_dataset.feat_mean, built.full_dataset.feat_std,
            n_in_channels, built.onset_prior, built.sustain_prior,
            args,
        )
        best_line = (f"  -> New best composite={best_composite:.3f} "
                     f"(onset_tol_f1={val_metrics['onset_tol_f1']:.3f} "
                     f"sustain_f1={val_metrics['sustain_f1']:.3f} "
                     f"intensity_auc={val_metrics['intensity_jump_auc']:.3f} "
                     f"arrow_col_f1={val_metrics.get('arrow_col_f1', 0.0):.3f})")
        print(best_line, flush=True)
        with open(EPOCH_LOG_PATH, 'a') as f:
            f.write(best_line + '\n')

print(f'\nTraining complete! Best composite={best_composite:.3f}')
print(f'Best model saved at: {CHECKPOINT_DIR}/best_model.pt')
print(f'Per-epoch log saved at: {EPOCH_LOG_PATH}')


## After Training (v8)

1. Download `best_model.pt` from Google Drive (`stepageddon/checkpoints/best_model.pt`).
2. Place it in `backend/ml/checkpoints/best_model.pt`.
3. It loads as `arch_version=8`; inference sets `use_learned_arrows=True` automatically
   and blends the learned direction behind the foot-flow playability constraints.
4. Verify locally (from `backend/`, venv active):
   ```bash
   PYTHONPATH=. python -m scripts.validate_charts --limit 8
   ```
   Expect the same near-zero violations as v7 (playability is guaranteed by the
   constraints regardless of what the head proposes), now with music-aware direction.
5. Optionally A/B against the old v7 checkpoint on the same songs (same audio → same
   seed) to judge whether the learned footing reads better before making it the default.
